# Import File sized pca [780,381], frf [780,9,381] and extract baseline UD

In [28]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

#save file name
file_name = 'original'

# Load the data
pca_frf_data = np.abs(np.load(f'PCA_Data_{file_name}\\PCA_Frf.npy'))
frf_data = np.abs(np.load(f'PCA_Data_{file_name}\\frf_data.npy'))
# output_dir = rf"..\450,60,2 Dataset\processed_dataset3"
# f = np.arange(10, 200, 0.5)  # Frequency vector from 10 to 200 Hz with 0.5 Hz resolution
f_max = 200  # Maximum frequency
num_points = frf_data.shape[2] # This is 381
freq_axis = np.linspace(10, f_max, num_points)

# Define your cases (Must match the order in your 780 rows)
case_names = ['UD', 'L1', 'L2', 'L3', 'L4', 'M1', 'M2', 'M3', 'M4', 'H1', 'H2', 'H3', 'H4']
all_locations = []
all_labels = np.repeat(case_names, 60) # Creates a list of 780 labels

# Create DataFrame for easier manipulation
pca_df = pd.DataFrame(pca_frf_data.T, columns=all_labels) 
# Note: We Transpose (.T) so frequency is rows and cases are columns to match your old logic

print("Frequency axis shape:", freq_axis.shape)
print("frequency axis:", freq_axis[375:381])
print ("frf_data shape:", frf_data.shape)
print ("PCA DataFrame shape:", pca_df.shape)
print ("PCA DataFrame columns:", pca_df.columns)

Frequency axis shape: (381,)
frequency axis: [197.5 198.  198.5 199.  199.5 200. ]
frf_data shape: (780, 9, 381)
PCA DataFrame shape: (381, 780)
PCA DataFrame columns: Index(['UD', 'UD', 'UD', 'UD', 'UD', 'UD', 'UD', 'UD', 'UD', 'UD',
       ...
       'H4', 'H4', 'H4', 'H4', 'H4', 'H4', 'H4', 'H4', 'H4', 'H4'],
      dtype='str', length=780)


In [29]:
# import numpy as np
# import matplotlib.pyplot as plt


# def plot_trial_by_freq(trial_index, x_limit_hz=200):
#     plt.figure(figsize=(12, 6))
    
#     # Calculate which index corresponds to the frequency limit
#     # This ensures we only plot what is needed
#     idx_limit = np.searchsorted(f, x_limit_hz)
    
#     # Loop through the 9 sensors
#     for sensor_id in range(frf_data.shape[1]):
#         # Plot using freq_axis as the X-axis
#         plt.plot(f[:idx_limit], 
#                  frf_data[trial_index, sensor_id, :idx_limit], 
#                  label=f'Sensor {sensor_id + 1}')

#     # Formatting
#     plt.title(f'FRF Magnitude - Trial {trial_index}', fontsize=14)
#     plt.xlabel('Frequency (Hz)', fontsize=12)
#     plt.ylabel('Magnitude', fontsize=12)
    
#     plt.xlim(30, 50) # Limit the view to 200 Hz
#     plt.legend(loc='upper right', ncol=3)
#     plt.grid(True, which='both', linestyle='--', alpha=0.5)
#     plt.tight_layout()
#     plt.show()

# # --- Run the plot ---
# # Set x_limit_hz to 200 to see the first 200 Hz
# plot_trial_by_freq(trial_index=0, x_limit_hz=200)

# the frf_data is in the shape of (780, 9, 2049) 
# mask = (f >= 30) & (f <= 50)
# frf_data[0,:, mask].shape
# # the value should be (9, 41) which is 9 sensors and 41 frequency points between 30 and 50 Hz. 
# # but when i print the shape it gives me (41, 9) which is inverted.
# # if I transpose it, it gives me the correct shape of (9, 41). but is the output still correct?

Extract 60 UD data for mean baseline

In [30]:
# Extract all UD trials (first 60 columns)
ud_trials = pca_df['UD'] 
mean_ud_baseline = ud_trials.mean(axis=1)
print ("Mean UD Baseline shape:", mean_ud_baseline.shape)

# Calculate the slope (derivative) for WCC
s = pca_df.diff().shift(-1).fillna(0)
s_baseline = mean_ud_baseline.diff().shift(-1).fillna(0)

np.save(f'healthy_slope_baseline_{file_name}.npy', s_baseline.values) # Save the slope baseline for later use in the Digital Twin
print(s_baseline.shape)


# # Another method to calculate the slope baseline using np
# saved = np.load(r'D:\UM Document\FYP_ML\Machine Learning Algorithm\healthy_slope_baseline_original.npy')
# print("Loaded slope baseline shape:", saved.shape)

# print("Loaded slope baseline:", saved[0:10]) # Print the first 10 values to verify
# print("Calculated slope baseline:", s_baseline.values[0:10]) # Print the first 10 values to verify

Mean UD Baseline shape: (381,)
(381,)


# Function

In [31]:
def extract_pca_features(f_array, pca_df, s_df, s_baseline, labels, f_low, f_high):
    # Calculate integer bounds
    idx_start = np.searchsorted(f_array, f_low)
    idx_end = np.searchsorted(f_array, f_high, side='right')
    
    # Slice using integer positions (.iloc for DataFrames)
    # pca_slope
    f_win = f_array[idx_start:idx_end]
    pca_win = pca_df.iloc[idx_start:idx_end]

    # undamaged slope window for WCC
    s_win = s_df.iloc[idx_start:idx_end] #what is s_df usage
    s_base_win = s_baseline.iloc[idx_start:idx_end]
    
    # Check if window is empty
    if len(f_win) == 0:
        return pd.DataFrame()

    # Pre-calculate baseline values
    s_base_max = s_base_win.abs().max()
    if pd.isna(s_base_max) or s_base_max == 0: s_base_max = 1
    s_rs_base_vals = (s_base_win * 50 / s_base_max).values
    
    results = []
    for i in range(pca_win.shape[1]):
        win_pca_vals = pca_win.iloc[:, i].values
        win_slope_vals = s_win.iloc[:, i].values
        
        # WCC Calculation
        s_max = np.abs(win_slope_vals).max()
        if pd.isna(s_max) or s_max == 0: s_max = 1
        s_rs_curr = win_slope_vals * 50 / s_max
        wcc = np.sum(np.abs(s_rs_curr - s_rs_base_vals))
        
        # Peak detection (argmax is the safest fallback)
        peaks, _ = find_peaks(win_pca_vals, prominence=0.01)
        if peaks.size > 0:
            best_idx = peaks[np.argmax(win_pca_vals[peaks])]
        else:
            best_idx = np.argmax(win_pca_vals)
            
        results.append({
            'case': labels[i], 
            'wcc': wcc, 
            'pca_peak_amp': win_pca_vals[best_idx], 
            'pca_peak_freq': f_win[best_idx]
        })
    print(f"Rescaled slope max: {s_rs_curr.max():.2f}, Rescaled baseline max: {s_rs_base_vals.max():.2f}")

    return pd.DataFrame(results)

In [32]:
# def extract_point_features(f_array, sensor_data, labels, f_low, f_high):
#     # Calculate integer bounds
#     idx_start = np.searchsorted(f_array, f_low)
#     idx_end = np.searchsorted(f_array, f_high, side='right')
#     mask = (f_array >= f_low) & (f_array <= f_high)
#     results = []
#     testsize = sensor_data[:, :, idx_start:idx_end]
#     print ("Trial sensors shape:", testsize.shape) 
    
#     for i in range(sensor_data.shape[0]):
#         trial_sensors = sensor_data[i, :, mask]
#         if trial_sensors.shape == (41, 9):
#             trial_sensors = trial_sensors.T
#             # print ("Shape inverted, now trial sensors shape:", trial_sensors.shape)
#         point_data = {'case': labels[i]}
            
#         for pt_idx in range(9):
#             sensor_series = trial_sensors[pt_idx, :]
            
#             # Use a prominence filter! 
#             # This prevents the code from catching tiny bumps at the 1.9 level
#             peaks, props = find_peaks(sensor_series, prominence=1.5) 
            
#             if peaks.size > 0:
#                 # This specifically finds the peak with the MAXIMUM amplitude
#                 highest_peak_idx = peaks[np.argmax(sensor_series[peaks])]
#                 p_amp = sensor_series[highest_peak_idx]
#             else:
#                 # If no peak is 'prominent' enough, just take the max value in range
#                 p_amp = np.max(sensor_series)
                
#             point_data[f'pt{pt_idx+1}_peak_amp'] = p_amp
            
#         results.append(point_data)      
        
#     return pd.DataFrame(results)    

In [33]:
def extract_point_features(f_array, sensor_data, labels, f_low, f_high):
    idx_start = np.searchsorted(f_array, f_low)
    idx_end = np.searchsorted(f_array, f_high, side='right')
    results = []

    for i in range(sensor_data.shape[0]):
        trial_sensors = sensor_data[i, :, idx_start:idx_end]
        point_data = {'case': labels[i]}

        for pt_idx in range(9):
            sensor_series = trial_sensors[pt_idx, :]  # slice column, not row

            peaks, props = find_peaks(sensor_series, prominence=1.5)

            if peaks.size > 0:
                highest_peak_idx = peaks[np.argmax(sensor_series[peaks])]
                p_amp = sensor_series[highest_peak_idx]
            else:
                p_amp = np.max(sensor_series)

            point_data[f'pt{pt_idx+1}_peak_amp'] = p_amp

        results.append(point_data)

    return pd.DataFrame(results)

# Mode 1

In [34]:
# --- MODE 1 (e.g., 20-60 Hz) ---
freq_lower1, freq_upper1 = 20, 60   

# Paste this BEFORE calling extract_wcc_feature in both pipelines
idx_start = np.searchsorted(freq_axis, freq_lower1)
idx_end = np.searchsorted(freq_axis, freq_upper1, side='right')
print("pca_frf dtype:", type(pca_df), getattr(pca_df, 'shape', None))
print("s_baseline dtype:", type(s_baseline), getattr(s_baseline, 'shape', None))
print("s_baseline[:5]:", s_baseline[:5])          # first 5 values — are they the same?
print("s_baseline global max:", np.abs(s_baseline).max())
print("freqs[idx_start:idx_start+3]:", freq_axis[idx_start:idx_start+3])  # window start matches?
print("freqs[idx_end-3:idx_end]:", freq_axis[idx_end-3:idx_end])          # window end matches?


pca_m1 = extract_pca_features(freq_axis, pca_df, s, s_baseline, all_labels, freq_lower1, freq_upper1)
pts_m1 = extract_point_features(freq_axis, frf_data, all_labels, freq_lower1, freq_upper1)

ratios = pd.DataFrame()
ratios['pt2/8'] = pts_m1['pt2_peak_amp'] / pts_m1['pt8_peak_amp']
ratios['pt4/6'] = pts_m1['pt4_peak_amp'] / pts_m1['pt6_peak_amp']
ratios['pt1/3'] = pts_m1['pt1_peak_amp'] / pts_m1['pt3_peak_amp']
ratios['pt3/9'] = pts_m1['pt3_peak_amp'] / pts_m1['pt9_peak_amp']
ratios['pt1/7'] = pts_m1['pt1_peak_amp'] / pts_m1['pt7_peak_amp']  
ratios['pt7/9'] = pts_m1['pt7_peak_amp'] / pts_m1['pt9_peak_amp']

print("Mode 1 PCA Features:\n", pca_m1.head()) 
print("Mode 1 Point Features:\n", pts_m1.head())
print("Mode 1 Ratios:\n", ratios.head())  

# print("Mode 1 Features:\n", pca_m1.iloc[55:60])  # Print a few rows for Mode 1 first case
# print("Mode 1 Features:\n", pts_m1.iloc[115:120])  # Print a few rows for Mode 1 second case
# print("Mode 1 Features:\n", pts_m1.iloc[175:180])  # Print a few rows for Mode 1 third case
# print("Mode 1 Features:\n", pts_m1.iloc[235:240])  # Print a few rows for Mode 1 fourth case
# print("Mode 1 Features:\n", pts_m1.iloc[295:300])  # Print a few rows for Mode 1 fifth case
# print("Mode 1 Features:\n", pts_m1.iloc[355:360])  # Print a few rows for Mode 1 sixth case
# print("Mode 1 Features:\n", pts_m1.iloc[415:420])  # Print a few rows for Mode 1 seventh case
# print("Mode 1 Features:\n", pts_m1.iloc[475:480])  # Print a few rows for Mode 1 eighth case
# print("Mode 1 Features:\n", pts_m1.iloc[535:540])  # Print a few rows for Mode 1 ninth case
# print("Mode 1 Features:\n", pts_m1.iloc[595:600])  # Print a few rows for Mode 1 tenth case
# print("Mode 1 Features:\n", pts_m1.iloc[655:660])  # Print a few rows for Mode 1 eleventh case
# print("Mode 1 Features:\n", pts_m1.iloc[715:720])  # Print a few rows for Mode 1 twelfth case
# print("Mode 1 Features:\n", pts_m1.iloc[775:780])  # Print a few rows for Mode 1 thirteenth case


pca_frf dtype: <class 'pandas.DataFrame'> (381, 780)
s_baseline dtype: <class 'pandas.Series'> (381,)
s_baseline[:5]: 0    0.015977
1    0.016784
2    0.016103
3    0.018639
4    0.017062
dtype: float64
s_baseline global max: 4.0543094671136615
freqs[idx_start:idx_start+3]: [20.  20.5 21. ]
freqs[idx_end-3:idx_end]: [59.  59.5 60. ]
Rescaled slope max: 50.00, Rescaled baseline max: 50.00
Mode 1 PCA Features:
   case        wcc  pca_peak_amp  pca_peak_freq
0   UD  35.899241     26.395564           43.5
1   UD  31.472383     27.836757           43.5
2   UD  28.045078     27.561522           43.5
3   UD  26.961665     27.666205           43.5
4   UD  26.437170     28.323495           43.5
Mode 1 Point Features:
   case  pt1_peak_amp  pt2_peak_amp  pt3_peak_amp  pt4_peak_amp  pt5_peak_amp  \
0   UD      4.867258      4.885259      4.700070     16.059352     15.596533   
1   UD      5.061269      5.078546      4.874012     16.694105     16.211473   
2   UD      5.024621      5.044684      4

# Mode 2

In [35]:
# --- MODE 2 (e.g., 90-120 Hz) ---
freq_lower2, freq_upper2 = 90, 120
pca_m2 = extract_pca_features(freq_axis, pca_df, s, s_baseline, all_labels, freq_lower2, freq_upper2)
# pts_m2 = extract_point_features(freq_axis, frf_data, all_labels, freq_lower2, freq_upper2)

print("Mode 2 PCA Features:\n", pca_m2.head()) 
# print("Mode 2 Point Features:\n", pts_m2.head())  

Rescaled slope max: 50.00, Rescaled baseline max: 50.00
Mode 2 PCA Features:
   case        wcc  pca_peak_amp  pca_peak_freq
0   UD  97.338088     22.043468          110.0
1   UD  83.077181     22.585146          110.0
2   UD  88.548694     22.244791          110.0
3   UD  90.231843     22.065231          110.0
4   UD  86.030551     22.548783          110.0


# Mode 3

In [36]:
# --- MODE 3 (e.g., 130-160 Hz) ---
freq_lower3, freq_upper3 = 130, 160
pca_m3 = extract_pca_features(freq_axis, pca_df, s, s_baseline, all_labels, freq_lower3, freq_upper3)
# pts_m3 = extract_point_features(freq_axis, frf_data, all_labels, freq_lower3, freq_upper3)

print("Mode 3 PCA Features:\n", pca_m3.head()) 
# print("Mode 3 Point Features:\n", pts_m3.head())  

Rescaled slope max: -1.40, Rescaled baseline max: 37.88
Mode 3 PCA Features:
   case         wcc  pca_peak_amp  pca_peak_freq
0   UD  221.427529     16.442444          147.0
1   UD  181.594591     16.312216          147.0
2   UD  221.174144     16.203430          146.5
3   UD  223.875167     16.189414          146.5
4   UD  209.294255     16.103828          146.5


# Merge and Save File

In [37]:
# 1. Prepare raw combined modes with suffixes
# We add suffixes to EVERY column first so we can pick them easily
m1_raw = pd.concat([pca_m1, pts_m1.drop(columns=['case'])], axis=1).add_suffix('_m1')
# m2_raw = pd.concat([pca_m2, pts_m2.drop(columns=['case', 'severity', 'damage_location'])], axis=1).add_suffix('_m2')
# m3_raw = pd.concat([pca_m3, pts_m3.drop(columns=['case', 'severity', 'damage_location'])], axis=1).add_suffix('_m3')
m2_raw = pca_m2.add_suffix('_m2')  # Only PCA features for Mode 2
m3_raw = pca_m3.add_suffix('_m3')  # Only PCA features for Mode 3

# 2. Extract the Labels (using Mode 1 as the reference)
# We rename them back to clean names (no suffix)
labels = pca_m1[['case']].copy()

# 3. Group the WCC columns together
wcc_all_modes = pd.DataFrame({
    'wcc_m1': m1_raw['wcc_m1'],
    'wcc_m2': m2_raw['wcc_m2'],
    'wcc_m3': m3_raw['wcc_m3']
})

# 4. Group the PCA Peak info together
pca_peaks_all = pd.concat([
    m1_raw[['pca_peak_amp_m1', 'pca_peak_freq_m1']],
    m2_raw[['pca_peak_amp_m2', 'pca_peak_freq_m2']],
    m3_raw[['pca_peak_amp_m3', 'pca_peak_freq_m3']]
], axis=1)

# 5. Group the Sensor Points (pt1 to pt9)
# We collect all 'pt' columns from each mode
pts_m1 = m1_raw[[c for c in m1_raw.columns if c.startswith('pt')]]
pts_m2 = m2_raw[[c for c in m2_raw.columns if c.startswith('pt')]]
pts_m3 = m3_raw[[c for c in m3_raw.columns if c.startswith('pt')]]


# 6. Final Concatenation: Reordering everything
final_feature_matrix = pd.concat([
    labels, 
    wcc_all_modes, 
    pca_peaks_all, 
    pts_m1, 
    pts_m2, 
    pts_m3,
    ratios

], axis=1)

# 7. Save to CSV
final_feature_matrix.to_csv(f'master_ema_features_{file_name}.csv', index=False)

print("Success! Master file created with organized columns.")
print(f"Columns: {final_feature_matrix.columns.tolist()[:10]} ...") # Shows the first 10 headers

Success! Master file created with organized columns.
Columns: ['case', 'wcc_m1', 'wcc_m2', 'wcc_m3', 'pca_peak_amp_m1', 'pca_peak_freq_m1', 'pca_peak_amp_m2', 'pca_peak_freq_m2', 'pca_peak_amp_m3', 'pca_peak_freq_m3'] ...


In [38]:
def prepare_plotting_labels(df):
    def get_sev_loc(case_str):
        # Force to string to prevent 'float' errors
        case_str = str(case_str)
        
        if case_str == 'UD':
            return 'UD', '0'
        
        # 1. Map Severity (First Letter)
        sev_map = {'L': 'LD', 'M': 'MD', 'H': 'HD'}
        sev = sev_map.get(case_str[0], 'Other')
        
        # 2. Map Location (Last Character)
        # If case is 'L1', case_str[-1] is '1'
        loc = case_str[-1]
        
        return sev, loc

    # 'case' is the column containing UD, L1, M2, etc.
    df['severity_plot'], df['damage_location'] = zip(*df['case'].map(get_sev_loc))
    
    # Categorical order so plots show UD -> LD -> MD -> HD
    df['severity_plot'] = pd.Categorical(df['severity_plot'], 
                                         categories=['UD', 'LD', 'MD', 'HD'], 
                                         ordered=True)
    return df

# Apply to your master matrix
final_feature_matrix = prepare_plotting_labels(final_feature_matrix)

final_feature_matrix.to_csv(f'pca_features_{file_name}.csv', index=False)

# Graph Plotting

In [39]:
import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns
import matplotlib.pyplot as plt

def plot_wcc_master(df, mode_num):
    plt.figure(figsize=(10, 6))
    
    # y-axis is the WCC column for that mode (e.g., wcc_m1)
    sns.barplot(x='severity_plot', 
                y=f'wcc_m{mode_num}', 
                hue='damage_location', 
                data=df, 
                errorbar='sd',
                palette='viridis')
    
    plt.title(f'Global Damage Indicator (WCC) - Mode {mode_num}')
    plt.xlabel('Damage Severity')
    plt.ylabel(f'WCC Value ($a_{{s{mode_num}}}$)')
    plt.legend(title='Damage Location', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

# # Run it for Mode 1
# plot_wcc_master(final_feature_matrix, 1)
# plot_wcc_master(final_feature_matrix, 2)
# plot_wcc_master(final_feature_matrix, 3)



In [40]:
# def plot_sensor_response(df, mode_num, sensor_num):
#     plt.figure(figsize=(10, 6))
    
#     # y-axis is the specific sensor peak (e.g., pt1_peak_amp_m1)
#     y_col = f'pt{sensor_num}_peak_amp_m{mode_num}'
    
#     sns.barplot(x='severity_plot', 
#                 y=y_col, 
#                 hue='damage_location', 
#                 data=df, 
#                 errorbar='sd',
#                 palette='magma')
    
#     plt.title(f'Sensor {sensor_num} Amplitude Response - Mode {mode_num}')
#     plt.xlabel('Damage Severity')
#     plt.ylabel('Peak Amplitude (g/N)')
#     plt.legend(title='Damage Site', bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.grid(axis='y', alpha=0.3)
#     plt.show()

# # (mode number and sensor number are 1-indexed for readability)
# for i in range(1, 10):
#     plot_sensor_response(final_feature_matrix, 1, i)